# Complex-Time MAIN: 1, 3, or 5 bounces

Same trainer as `Complex_Time_QHO_Anharmonic_MU_lambda0_0p01_T2to4.ipynb`.
The second bounce pair is the search from `Complex_Time_QHO_Anharmonic_MU_Time_Loop_5_thimble _Inv.ipynb`.

**Edit the next cell, then run the notebook from the top.**

| Switch | What is trained | Result |
|---|---|---|
| `RUN_1_BOUNCE` | zero saddle only | \(K = K_0\) |
| `RUN_3_BOUNCES` | zero saddle + one opposite pair | \(K = K_0 + 2 K_{\mathrm{bounce}}\) |
| `RUN_5_BOUNCES` | those three + a second opposite pair | \(K = K_3 + 2 K_{\mathrm{bounce2}}\) |

Set a switch to `False` to turn that count off. To focus on one count, leave only that switch `True`.

Five bounces still trains the zero saddle and the first pair, because the five-bounce sum is built from them. The saved answer is the five-bounce sum.

`LAMBDA_LIST` is the lambda loop. `T_START`, `T_STOP`, and `DT` build the time loop. Replace `T_LIST` with your own times if you want a custom list.

Each \((\lambda, T)\) is saved as soon as it finishes. Run the training cell again to continue. Rows already marked OK are skipped.


In [ ]:
# ============================================================
# EDIT THIS CELL. This chooses the whole run.
# ============================================================

# This check runs these pairs only. Set JOBS = None to use every lambda with every time.
LAMBDA_LIST = [0.1, 0.3]
#T_LIST = [2.2]
#JOBS = [(0.1, 2.2), (0.3, 2.2)]

# Full loop, unused while JOBS is set:
LAMBDA_LIST = [0.1] 

# LAMBDA_LIST = [0.01, 0.1, 0.3, 0.5]
T_START = 0.0
T_STOP = 6.0
DT = 0.1 
T_LIST = [round(T_START + i * DT, 2) for i in range(int(round((T_STOP - T_START) / DT)) + 1)]
JOBS = None 

# Which bounce counts to train.
# True  = train this count and put it in the table and the plot.
# False = skip it.
# Leave only one True to focus on that count.
RUN_1_BOUNCE = True     # zero saddle only
RUN_3_BOUNCES = True    # zero saddle + one opposite pair
RUN_5_BOUNCES = True    # those three + a second opposite pair


## Setup

Same \(N=16\), \(\eta=0.05\), rank, epochs, and batch as the T = 2 to 4 notebook. Leave `QUICK_TEST` False.


In [2]:
from __future__ import annotations

import csv
import json
import math
import time
import traceback
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from scipy.optimize import root

%matplotlib inline

torch.set_default_dtype(torch.float64)

if not (RUN_1_BOUNCE or RUN_3_BOUNCES or RUN_5_BOUNCES):
    raise RuntimeError("Turn on at least one of RUN_1_BOUNCE, RUN_3_BOUNCES, RUN_5_BOUNCES.")

# Physics: same N=16 complex-time lattice as exact_check_Time_Loop.py
N = 16
D = N - 1
ETA = 0.05
MASS = 1.0
OMEGA = 1.0
LAMBDA = float(LAMBDA_LIST[0])
T = float(T_LIST[0])
A_C = (T - 1j * ETA) / N
X_I = X_F = 0.0

EXACT_DIR = Path("Re__Complex_Time_Code_") / "py_scripts_extracted"
EXACT_CSVS = {
    0.0: EXACT_DIR / "exact_check_Time_Loop_lambda0_T0to8_dt0.01.csv",
    0.01: EXACT_DIR / "exact_check_Time_Loop_lambda0p01_T0to8_dt0.01.csv",
    0.1: EXACT_DIR / "exact_check_Time_Loop_lambda0p1_T0to8_dt0.01.csv",
    0.5: EXACT_DIR / "exact_check_Time_Loop_lambda0.5_dt0.01.csv",
}

# Same training settings as Complex_Time_QHO_Anharmonic_MU_lambda0_0p01_T2to4.ipynb
RANK = 16
EPOCHS = 1_000
BATCH = 512
LEARNING_RATE = 1.0e-3
N_MC = 80_000
MC_CHUNK = 5_000
SEED = 52
LOG_EVERY = 250

# Set True only for a syntax check. Its numbers are not physics results.
QUICK_TEST = False
if QUICK_TEST:
    EPOCHS, BATCH, N_MC, MC_CHUNK, LOG_EVERY = 5, 32, 1_000, 500, 1

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loop_dir = Path("runs") / f"MAIN_Multihimble_N{N}"
loop_dir.mkdir(parents=True, exist_ok=True)
run_dir = loop_dir

print(f"device={DEVICE}; dtype=float64/complex128")
print(f"N={N}, D={D}, eta={ETA}")
print(f"lambda loop ({len(LAMBDA_LIST)}): {LAMBDA_LIST}")
print(f"time loop ({len(T_LIST)}): {T_LIST[0]} to {T_LIST[-1]}   list={T_LIST}")
print(f"1 bounce  (zero saddle only)              : {RUN_1_BOUNCE}")
print(f"3 bounces (zero + one pair)               : {RUN_3_BOUNCES}")
print(f"5 bounces (those + a second pair)         : {RUN_5_BOUNCES}")
if RUN_5_BOUNCES and not (RUN_1_BOUNCE or RUN_3_BOUNCES):
    print("Five bounces is the plotted answer. The zero saddle and the first pair are still trained, because the sum uses them.")
elif RUN_3_BOUNCES and not RUN_1_BOUNCE and not RUN_5_BOUNCES:
    print("This run focuses on 3 bounces.")
elif RUN_1_BOUNCE and not RUN_3_BOUNCES and not RUN_5_BOUNCES:
    print("This run focuses on 1 bounce: the zero saddle only. No bounce search.")
print(f"rank={RANK}, epochs={EPOCHS} per saddle, batch={BATCH}, N_mc={N_MC}")
print(f"results directory: {loop_dir.resolve()}")


device=cpu; dtype=float64/complex128
N=16, D=15, eta=0.05
lambda loop (1): [0.1]
time loop (61): 0.0 to 6.0   list=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0, 2.1, 2.2, 2.3, 2.4, 2.5, 2.6, 2.7, 2.8, 2.9, 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 4.0, 4.1, 4.2, 4.3, 4.4, 4.5, 4.6, 4.7, 4.8, 4.9, 5.0, 5.1, 5.2, 5.3, 5.4, 5.5, 5.6, 5.7, 5.8, 5.9, 6.0]
1 bounce  (zero saddle only)              : True
3 bounces (zero + one pair)               : True
5 bounces (those + a second pair)         : True
rank=16, epochs=1000 per saddle, batch=512, N_mc=80000
results directory: C:\xampp1\Quantum\runs\MAIN_Multihimble_N16


## 1. Lattice action

Same action, gradient, and Hessian as the T = 2 to 4 notebook. \(a_c = (T - i\eta)/N\), weight \(e^{iS}\).


In [3]:
def action_np(x: np.ndarray) -> complex:
    """Discretised complex-time action for one path."""
    padded = np.r_[0.0 + 0.0j, x, 0.0 + 0.0j]
    dx = np.diff(padded)
    return (
        MASS / (2.0 * A_C) * np.sum(dx**2)
        - A_C * MASS * OMEGA**2 / 2.0 * np.sum(x**2)
        - A_C * LAMBDA / 4.0 * np.sum(x**4)
    )


def gradient_np(x: np.ndarray) -> np.ndarray:
    """Holomorphic gradient dS/dx at the interior lattice sites."""
    padded = np.r_[0.0 + 0.0j, x, 0.0 + 0.0j]
    return (
        (MASS / A_C) * (2.0 * x - padded[:-2] - padded[2:])
        - A_C * MASS * OMEGA**2 * x
        - A_C * LAMBDA * x**3
    )


def hessian_np(x: np.ndarray) -> np.ndarray:
    """Complex-symmetric Hessian d^2S/dx_j dx_k at a saddle."""
    hessian = np.zeros((D, D), dtype=np.complex128)
    np.fill_diagonal(
        hessian,
        2.0 * MASS / A_C
        - A_C * MASS * OMEGA**2
        - 3.0 * A_C * LAMBDA * x**2,
    )
    np.fill_diagonal(hessian[1:], -MASS / A_C)
    np.fill_diagonal(hessian[:, 1:], -MASS / A_C)
    return hessian


def packed_gradient(y: np.ndarray) -> np.ndarray:
    x = y[:D] + 1j * y[D:]
    grad = gradient_np(x)
    return np.r_[grad.real, grad.imag]


def action_torch(x: torch.Tensor) -> torch.Tensor:
    """Batch action accumulated entirely in complex128."""
    x = x.to(torch.complex128)
    zero = torch.zeros((x.shape[0], 1), dtype=torch.complex128, device=x.device)
    padded = torch.cat((zero, x, zero), dim=1)
    dx = padded[:, 1:] - padded[:, :-1]
    a = torch.as_tensor(A_C, dtype=torch.complex128, device=x.device)
    return (
        MASS / (2.0 * a) * torch.sum(dx**2, dim=1)
        - a * MASS * OMEGA**2 / 2.0 * torch.sum(x**2, dim=1)
        - a * LAMBDA / 4.0 * torch.sum(x**4, dim=1)
    )


## 2. Bounce search

The first bounce is the original T = 2 to 4 search: harmonic-1 sine, amplitudes from 0.5 to 10.

The second bounce is the five-thimble search: harmonics 2, then 3, then 1, dropping copies of the first bounce. It runs only when `RUN_5_BOUNCES` is True.


In [4]:
def first_bounce() -> np.ndarray:
    """Find the lowest-action nonzero zero-endpoint saddle.

    A hard quartic oscillator has amplitude-dependent frequency, so this solution exists
    even though T < pi/omega.  Several amplitudes are tried to avoid accepting x=0.
    """
    sites = np.arange(1, N)
    shape = np.sin(np.pi * sites / N)
    candidates = []
    for amplitude in np.linspace(0.5, 10.0, 20):
        guess = np.r_[amplitude * shape, np.zeros(D)]
        solved = root(packed_gradient, guess, method="lm", options={"maxiter": 5_000})
        x = solved.x[:D] + 1j * solved.x[D:]
        residual = np.max(np.abs(gradient_np(x)))
        if residual < 1e-8 and np.max(np.abs(x)) > 1e-3:
            if not any(np.max(np.abs(x - old)) < 1e-6 for old in candidates):
                candidates.append(x)
    if not candidates:
        raise RuntimeError("Could not find the first nonzero saddle")
    return min(candidates, key=lambda x: abs(action_np(x)))


def find_bounce_harmonic(harmonic: int) -> list:
    """Second-bounce search copied from the five-thimble notebook."""
    sites = np.arange(1, N)
    shape = np.sin(harmonic * np.pi * sites / N)
    candidates = []
    for amplitude in np.linspace(0.4, 12.0, 30):
        guess = np.r_[amplitude * shape, np.zeros(D)]
        solved = root(packed_gradient, guess, method="lm", options={"maxiter": 5_000})
        x = solved.x[:D] + 1j * solved.x[D:]
        residual = np.max(np.abs(gradient_np(x)))
        if residual < 1e-8 and np.max(np.abs(x)) > 1e-3:
            if not any(np.max(np.abs(x - old)) < 1e-6 for old in candidates):
                if not any(np.max(np.abs(x + old)) < 1e-6 for old in candidates):
                    candidates.append(x)
    return candidates


def second_bounce(first: np.ndarray):
    """A bounce distinct from the first pair. None if the search finds nothing."""
    pool = find_bounce_harmonic(2) + find_bounce_harmonic(3) + find_bounce_harmonic(1)
    distinct = []
    for x in pool:
        if np.max(np.abs(x - first)) < 1e-3 or np.max(np.abs(x + first)) < 1e-3:
            continue
        if any(np.max(np.abs(x - old)) < 1e-5 or np.max(np.abs(x + old)) < 1e-5 for old in distinct):
            continue
        distinct.append(x)
    if not distinct:
        return None
    return min(distinct, key=lambda x: action_np(x).imag)


def tangent_map(hessian: np.ndarray) -> np.ndarray:
    """Return canonically oriented A with A.T @ H @ A = iI."""
    values, vectors = np.linalg.eig(hessian)
    for k in range(D):
        vectors[:, k] /= np.sqrt(vectors[:, k].T @ vectors[:, k])

    # Complex orthogonality implies det(V)=+/-1.  Fix +1 so every saddle uses
    # the same orientation inherited from the real coordinate basis; otherwise
    # a LAPACK eigenvector sign arbitrarily changes K_sigma -> -K_sigma.
    if np.linalg.det(vectors).real < 0.0:
        vectors[:, 0] *= -1.0

    orthogonal_error = np.max(np.abs(vectors.T @ vectors - np.eye(D)))
    orientation_error = abs(np.linalg.det(vectors) - 1.0)
    tangent = vectors @ np.diag(np.sqrt(1j / values))
    residual = np.max(np.abs(tangent.T @ hessian @ tangent - 1j * np.eye(D)))
    if orthogonal_error > 1e-8 or orientation_error > 1e-8 or residual > 1e-8:
        raise RuntimeError(
            "Bad saddle tangent: "
            f"orthogonal={orthogonal_error}, orientation={orientation_error}, "
            f"congruence={residual}"
        )
    return tangent


def gaussian_saddle_estimate(saddle: np.ndarray) -> complex:
    """Quadratic saddle estimate; used only as a decomposition diagnostic."""
    hessian = hessian_np(saddle)
    norm = (MASS / (2.0 * np.pi * 1j * A_C)) ** (N / 2)
    return complex(
        norm * np.exp(1j * action_np(saddle))
        * (2.0 * np.pi * 1j) ** (D / 2) / np.sqrt(np.linalg.det(hessian))
    )


## 3. Saddle-anchored neural contour

Same `SaddleFlow` as the T = 2 to 4 notebook. The saddle and the tangent stay fixed. Only the rank-16 correction is trained.


In [5]:
class SaddleFlow(nn.Module):
    """Fixed saddle/tangent plus a saddle-anchored low-rank correction."""

    def __init__(self, saddle: np.ndarray, tangent: np.ndarray, seed: int):
        super().__init__()
        torch.manual_seed(seed)
        self.register_buffer(
            "saddle", torch.tensor(saddle, dtype=torch.complex128, device=DEVICE)
        )
        self.register_buffer(
            "A", torch.tensor(tangent, dtype=torch.complex128, device=DEVICE)
        )
        self.W1 = nn.Parameter(torch.randn(RANK, D, device=DEVICE) / math.sqrt(D))
        qmat, _ = torch.linalg.qr(torch.randn(RANK, RANK, device=DEVICE))
        self.W2 = nn.Parameter(qmat)
        # Tiny rather than exactly zero so every layer receives a gradient on step one.
        self.C_real = nn.Parameter(1e-5 * torch.randn(D, RANK, device=DEVICE))
        self.C_imag = nn.Parameter(1e-5 * torch.randn(D, RANK, device=DEVICE))

    def C(self) -> torch.Tensor:
        return self.C_real.to(torch.complex128) + 1j * self.C_imag.to(torch.complex128)

    def forward(self, z: torch.Tensor, return_cache: bool = False):
        z = z.to(dtype=torch.float64, device=DEVICE)
        a1 = torch.tanh(z @ self.W1.T)
        a2 = torch.tanh(a1 @ self.W2.T)
        x = (
            z.to(torch.complex128) @ self.A.T
            + self.saddle
            + a2.to(torch.complex128) @ self.C().T
        )
        return (x, (a1, a2)) if return_cache else x

    def full_jacobian(self, a1: torch.Tensor, a2: torch.Tensor) -> torch.Tensor:
        """Exact batch Jacobian A + C D2 W2 D1 W1, including all cross terms."""
        d1 = 1.0 - a1**2
        d2 = 1.0 - a2**2
        h1 = d1.unsqueeze(-1) * self.W1.unsqueeze(0)
        h2 = torch.matmul(self.W2.unsqueeze(0), h1)
        dh_dz = d2.unsqueeze(-1) * h2
        correction = torch.matmul(self.C().unsqueeze(0), dh_dz.to(torch.complex128))
        return self.A.unsqueeze(0) + correction

    def determinant_parts(self, a1: torch.Tensor, a2: torch.Tensor):
        """Return log|det J| and det(J)/|det(J)| via the corrected low-rank formula."""
        sign_a, logabs_a = torch.linalg.slogdet(self.A)
        # Correct for forward x = z @ A.T.  This is A^{-1}C, not A^{-T}C.
        k_mat = self.W1.to(torch.complex128) @ torch.linalg.solve(self.A, self.C())
        d1 = (1.0 - a1**2).to(torch.complex128)
        d2 = (1.0 - a2**2).to(torch.complex128)
        m1 = d1.unsqueeze(-1) * k_mat.unsqueeze(0)
        m2 = torch.matmul(self.W2.to(torch.complex128).unsqueeze(0), m1)
        correction = (
            torch.eye(RANK, dtype=torch.complex128, device=DEVICE).unsqueeze(0)
            + d2.unsqueeze(-1) * m2
        )
        sign_c, logabs_c = torch.linalg.slogdet(correction)
        return logabs_a + logabs_c, sign_a * sign_c


## 4. Preflight

Same residual, parity, tangent, Jacobian, and determinant checks as the T = 2 to 4 notebook. A bounce that fails the check is skipped. The zero saddle is still saved.


In [6]:
def validate_saddles_and_jacobian(trivial: np.ndarray, bounce: np.ndarray) -> None:
    residual = np.max(np.abs(gradient_np(bounce)))
    parity_action = abs(action_np(bounce) - action_np(-bounce))
    parity_hessian = np.max(np.abs(hessian_np(bounce) - hessian_np(-bounce)))
    if residual > 1e-8 or parity_action > 1e-10 or parity_hessian > 1e-10:
        raise RuntimeError("Saddle or parity preflight failed")

    tangent = tangent_map(hessian_np(bounce))
    model = SaddleFlow(bounce, tangent, SEED + 999)
    # Make the nonlinear correction large enough that the check tests it, not only A.
    with torch.no_grad():
        model.C_real.normal_(std=0.03)
        model.C_imag.normal_(std=0.03)

    z = torch.randn(3, D, dtype=torch.float64, device=DEVICE)
    x, (a1, a2) = model(z, return_cache=True)
    jacobian = model.full_jacobian(a1, a2)
    eps = 1e-6
    jacobian_fd = torch.empty_like(jacobian)
    with torch.no_grad():
        for k in range(D):
            dz = torch.zeros_like(z)
            dz[:, k] = eps
            jacobian_fd[:, :, k] = (model(z + dz) - model(z - dz)) / (2.0 * eps)
    fd_error = float(torch.max(torch.abs(jacobian - jacobian_fd)).cpu())

    logabs, sign = model.determinant_parts(a1, a2)
    det_low_rank = torch.exp(logabs) * sign
    det_full = torch.linalg.det(jacobian)
    determinant_error = float(torch.max(torch.abs(det_low_rank - det_full)).cpu())
    determinant_relative = determinant_error / max(
        float(torch.max(torch.abs(det_full)).cpu()), 1e-300
    )

    print(f"bounce residual              : {residual:.3e}")
    print(f"parity action/Hessian errors : {parity_action:.3e}, {parity_hessian:.3e}")
    print(f"full Jacobian FD error       : {fd_error:.3e}")
    print(f"low-rank determinant rel err : {determinant_relative:.3e}")
    if fd_error > 3e-7 or determinant_relative > 3e-8:
        raise RuntimeError("Corrected Jacobian preflight failed")
    print("All saddle and Jacobian preflight checks PASSED")


## 5. Training and Monte Carlo

Same loss and the same estimator as the T = 2 to 4 notebook.


In [7]:
LOG_2PI = math.log(2.0 * math.pi)


def weight_parts(model: SaddleFlow, z: torch.Tensor):
    """Branch-safe representation of exp(iS) det(J) / P(z)."""
    x, (a1, a2) = model(z, return_cache=True)
    action = action_torch(x)
    logabs_j, sign_j = model.determinant_parts(a1, a2)
    log_p = -0.5 * torch.sum(z**2, dim=1) - 0.5 * D * LOG_2PI
    logabs_w = -action.imag + logabs_j - log_p
    phase_w = torch.exp(1j * action.real) * sign_j
    return logabs_w, phase_w


def train_thimble(label: str, saddle: np.ndarray, seed: int):
    tangent = tangent_map(hessian_np(saddle))
    model = SaddleFlow(saddle, tangent, seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS, eta_min=LEARNING_RATE * 0.05
    )
    history = {"total": [], "logmag": [], "phase": []}
    started = time.time()

    for epoch in range(EPOCHS):
        optimizer.zero_grad(set_to_none=True)
        z = torch.randn(BATCH, D, dtype=torch.float64, device=DEVICE)
        logabs_w, phase_w = weight_parts(model, z)
        logmag_variance = torch.var(logabs_w, unbiased=False)
        circular_phase = 1.0 - torch.abs(torch.mean(phase_w))**2
        loss = logmag_variance + circular_phase
        if not torch.isfinite(loss):
            raise FloatingPointError(f"{label}: nonfinite loss at epoch {epoch + 1}")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        scheduler.step()

        history["total"].append(float(loss.detach().cpu()))
        history["logmag"].append(float(logmag_variance.detach().cpu()))
        history["phase"].append(float(circular_phase.detach().cpu()))
        if epoch == 0 or (epoch + 1) % LOG_EVERY == 0 or epoch + 1 == EPOCHS:
            print(
                f"{label:8s} epoch {epoch+1:4d}/{EPOCHS}: "
                f"loss={history['total'][-1]:.3e}, "
                f"logmag={history['logmag'][-1]:.3e}, "
                f"phase={history['phase'][-1]:.3e}, "
                f"elapsed={time.time()-started:.0f}s",
                flush=True,
            )
    return model, history, time.time() - started


def estimate_thimble(model: SaddleFlow, seed: int):
    """Chunked complex128 MC estimate with five independent stability blocks."""
    torch.manual_seed(seed + 100_000)
    logs, phases = [], []
    remaining = N_MC
    model.eval()
    while remaining:
        count = min(MC_CHUNK, remaining)
        z = torch.randn(count, D, dtype=torch.float64, device=DEVICE)
        with torch.no_grad():
            logabs, phase = weight_parts(model, z)
        logs.append(logabs.cpu())
        phases.append(phase.cpu())
        remaining -= count

    logabs = torch.cat(logs)
    phase = torch.cat(phases)
    shift = torch.max(logabs)
    shifted = torch.exp(logabs - shift) * phase
    norm = (MASS / (2.0 * np.pi * 1j * A_C)) ** (N / 2)
    scale = norm * math.exp(float(shift))
    estimate = complex(scale * torch.mean(shifted).numpy())

    centered = shifted - torch.mean(shifted)
    stderr = abs(scale) * math.sqrt(float(torch.mean(torch.abs(centered)**2)) / N_MC)
    ess = float(
        torch.abs(torch.mean(shifted))**2 / torch.mean(torch.abs(shifted)**2)
    )
    blocks = [
        complex(scale * torch.mean(block).numpy())
        for block in torch.tensor_split(shifted, 5)
    ]
    block_scatter = float(np.std(np.asarray(blocks, dtype=np.complex128)))
    return estimate, stderr, ess, block_scatter, blocks


## 6. One \((\lambda, T)\), then the loop

The lattice target is the same-\(N\) FFT. Existing exact-check files are reused. Any other lambda or time is filled with that same FFT and cached.

The training cell saves after every \((\lambda, T)\). Run it again to continue.


In [8]:
def fftconvolve(a, b, mode="valid"):
    """Same FFT convolution as exact_check_Time_Loop.py."""
    n = len(a) + len(b) - 1
    nfft = 1 << (n - 1).bit_length()
    full = np.fft.ifft(np.fft.fft(a, nfft) * np.fft.fft(b, nfft))[:n]
    return full[len(b) - 1 : len(a)]


def lattice_K_fft(lam_val, ac, L=10.0, M=2**15):
    """Same N=16 lattice FFT as exact_check_Time_Loop.py."""
    x = np.linspace(-L, L, M)
    h = x[1] - x[0]
    pref = np.sqrt(MASS / (2 * np.pi * 1j * ac))
    u = np.linspace(-2 * L, 2 * L, 2 * M - 1)
    link = pref * np.exp(1j * MASS * u**2 / (2 * ac))
    f = pref * np.exp(1j * MASS * x**2 / (2 * ac))
    w = np.exp(-1j * ac * (0.5 * MASS * OMEGA**2 * x**2 + 0.25 * lam_val * x**4))
    for _ in range(N - 1):
        f = h * fftconvolve(link, w * f, mode="valid")
    return complex(np.interp(0.0, x, f.real) + 1j * np.interp(0.0, x, f.imag))


def mehler_K00(Tc: complex) -> complex:
    """Continuum harmonic Mehler kernel at x_i = x_f = 0. Exact only for lambda = 0."""
    return np.sqrt(MASS * OMEGA / (2.0 * np.pi * 1j * np.sin(OMEGA * Tc)))


def _blank():
    return ""


def _z_fields(prefix, value):
    if value is None:
        return {f"{prefix}_re": _blank(), f"{prefix}_im": _blank()}
    return {f"{prefix}_re": value.real, f"{prefix}_im": value.imag}


def rel_err(guess, target):
    if guess is None or target is None or abs(target) == 0:
        return _blank()
    return abs(guess - target) / abs(target)


ROW_FIELDS = [
    "lambda", "T", "N", "eta",
    "run_1_bounce", "run_3_bounces", "run_5_bounces",
    "n_saddles", "has_bounce", "has_bounce2",
    "K1_re", "K1_im", "err1",
    "K3_re", "K3_im", "err3",
    "K5_re", "K5_im", "err5",
    "K_ML_re", "K_ML_im", "complex_rel_err_vs_lattice",
    "K_lattice_re", "K_lattice_im",
    "K_mehler_re", "K_mehler_im",
    "trivial_ESS_over_N", "bounce_ESS_over_N", "bounce2_ESS_over_N",
    "status", "note", "run_directory",
]


def job_key(lam, t):
    return (
        round(float(lam), 10),
        round(float(t), 5),
        bool(RUN_1_BOUNCE),
        bool(RUN_3_BOUNCES),
        bool(RUN_5_BOUNCES),
    )


def load_lattice():
    """CSV lattice where a file already has this (lambda, T). FFT for the rest."""
    table = {}
    continuum = {}
    for lam, path in EXACT_CSVS.items():
        if not path.exists():
            continue
        with path.open("r", encoding="utf-8") as handle:
            for row in csv.DictReader(handle):
                key = (round(float(row["lambda"]), 10), round(float(row["T"]), 5))
                table[key] = complex(float(row["lattice_K_re"]), float(row["lattice_K_im"]))
                if row.get("continuum_K_re"):
                    continuum[key] = complex(float(row["continuum_K_re"]), float(row["continuum_K_im"]))
    cache = loop_dir / "lattice_cache.csv"
    if cache.exists():
        with cache.open("r", encoding="utf-8") as handle:
            for row in csv.DictReader(handle):
                key = (round(float(row["lambda"]), 10), round(float(row["T"]), 5))
                table[key] = complex(float(row["K_re"]), float(row["K_im"]))
    missing = []
    for lam in LAMBDA_LIST:
        for t in T_LIST:
            key = (round(float(lam), 10), round(float(t), 5))
            if key not in table:
                missing.append((lam, t))
    if missing:
        print(f"Computing {len(missing)} lattice FFT points.", flush=True)
        new_file = not cache.exists()
        with cache.open("a", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=["lambda", "T", "K_re", "K_im"])
            if new_file:
                writer.writeheader()
            for lam, t in missing:
                ac = (t - 1j * ETA) / N
                value = lattice_K_fft(lam, ac)
                key = (round(float(lam), 10), round(float(t), 5))
                table[key] = value
                writer.writerow({"lambda": lam, "T": t, "K_re": value.real, "K_im": value.imag})
                handle.flush()
                print(f"  lattice lambda={lam} T={t:.2f}  {value:+.6f}", flush=True)
    return table, continuum


def point_dir(lam, t):
    name = f"lam{lam:g}_T{t:.2f}".replace(".", "p")
    path = loop_dir / name
    path.mkdir(parents=True, exist_ok=True)
    return path


def save_model(directory, label, trained, n_saddles):
    path = directory / f"{label}_model.pt"
    torch.save(
        {
            "state_dict": trained["model"].state_dict(),
            "saddle": trained["saddle"],
            "lambda": LAMBDA,
            "T": T,
            "N": N,
            "rank": RANK,
            "seed": trained["seed"],
            "label": label,
            "n_saddles": n_saddles,
        },
        path,
    )
    return str(path)


def train_saddle(label, saddle, seed):
    print("\n" + "=" * 84, flush=True)
    print(f"Training {label} thimble at lambda={LAMBDA}, T={T}", flush=True)
    print("=" * 84, flush=True)
    model, history, seconds = train_thimble(label, saddle, seed)
    estimate, stderr, ess, block_scatter, blocks = estimate_thimble(model, seed)
    print(f"K_{label}={estimate:+.10f}", flush=True)
    print(f"stderr={stderr:.3e}, ESS/N={ess:.6f}, block scatter={block_scatter:.3e}", flush=True)
    return {
        "model": model,
        "history": history,
        "seconds": seconds,
        "K": estimate,
        "stderr": stderr,
        "ESS": ess,
        "block_scatter": block_scatter,
        "blocks": blocks,
        "saddle": saddle,
        "seed": seed,
    }


def accept_bounce(saddle, label):
    """Keep a bounce only when the original preflight passes."""
    try:
        validate_saddles_and_jacobian(np.zeros(D, dtype=np.complex128), saddle)
        return True, ""
    except Exception as exc:
        print(f"{label} found, then the saddle check failed: {exc}", flush=True)
        print(f"{label} will not be added. The zero saddle is still kept.", flush=True)
        return False, f"{label} check failed: {exc}"


def run_one_time(t_value, lattice_ref):
    global T, A_C, run_dir
    T = float(t_value)
    A_C = (T - 1j * ETA) / N
    run_dir = point_dir(LAMBDA, T)
    notes = []
    trivial_saddle = np.zeros(D, dtype=np.complex128)

    want_first = RUN_3_BOUNCES or RUN_5_BOUNCES
    bounce_saddle = None
    bounce2_saddle = None
    if want_first:
        try:
            bounce_saddle = first_bounce()
        except Exception as exc:
            print(f"No first bounce at lambda={LAMBDA}, T={T}: {exc}", flush=True)
            notes.append("no first bounce")
            bounce_saddle = None
        if bounce_saddle is not None:
            ok, note = accept_bounce(bounce_saddle, "first bounce")
            if not ok:
                notes.append(note)
                bounce_saddle = None
    if RUN_5_BOUNCES and bounce_saddle is not None:
        try:
            bounce2_saddle = second_bounce(bounce_saddle)
        except Exception as exc:
            print(f"Second-bounce search failed at lambda={LAMBDA}, T={T}: {exc}", flush=True)
            notes.append(f"second-bounce search failed: {exc}")
            bounce2_saddle = None
        if bounce2_saddle is None and not any(str(n).startswith("second-bounce") for n in notes):
            print(f"No second bounce at lambda={LAMBDA}, T={T}", flush=True)
            notes.append("no second bounce")
        elif bounce2_saddle is not None:
            ok, note = accept_bounce(bounce2_saddle, "second bounce")
            if not ok:
                notes.append(note)
                bounce2_saddle = None

    trained = {}
    plan = [("trivial", trivial_saddle, SEED)]
    if bounce_saddle is not None:
        plan.append(("bounce", bounce_saddle, SEED + 1))
    if bounce2_saddle is not None:
        plan.append(("bounce2", bounce2_saddle, SEED + 2))
    for label, saddle, seed in plan:
        try:
            trained[label] = train_saddle(label, saddle, seed)
        except Exception as exc:
            traceback.print_exc()
            notes.append(f"{label} training failed: {exc}")
            print(f"{label} training failed: {exc}", flush=True)

    if "trivial" not in trained:
        raise RuntimeError("zero saddle did not train; " + " | ".join(notes))

    k1 = trained["trivial"]["K"]
    k_bounce = trained["bounce"]["K"] if "bounce" in trained else None
    k_bounce2 = trained["bounce2"]["K"] if "bounce2" in trained else None
    k3 = k1 + 2.0 * k_bounce if k_bounce is not None else None
    k5 = k3 + 2.0 * k_bounce2 if k3 is not None and k_bounce2 is not None else None

    # The reported answer is the highest count that was both requested and obtained.
    if RUN_5_BOUNCES and k5 is not None:
        k_ml, n_saddles = k5, 5
    elif (RUN_3_BOUNCES or RUN_5_BOUNCES) and k3 is not None:
        k_ml, n_saddles = k3, 3
    else:
        k_ml, n_saddles = k1, 1

    for label, result in trained.items():
        save_model(run_dir, label, result, n_saddles)
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        for axis, key, title in zip(
            axes,
            ("total", "logmag", "phase"),
            ("total loss", "Var[log |W|]", "circular phase variance"),
        ):
            axis.semilogy(np.maximum(result["history"][key], 1e-18), lw=0.8)
            axis.set_title(title)
            axis.set_xlabel("epoch")
            axis.grid(alpha=0.3)
        fig.suptitle(f"{label}, lambda={LAMBDA}, T={T}, N={N}")
        fig.tight_layout()
        fig.savefig(run_dir / f"{label}_loss.png", dpi=150, bbox_inches="tight")
        plt.close(fig)

    if bounce_saddle is not None and "bounce" in trained:
        np.savez(
            run_dir / "bounce_saddle.npz",
            x=bounce_saddle,
            action=action_np(bounce_saddle),
            hessian=hessian_np(bounce_saddle),
        )
    if bounce2_saddle is not None and "bounce2" in trained:
        np.savez(
            run_dir / "bounce2_saddle.npz",
            x=bounce2_saddle,
            action=action_np(bounce2_saddle),
            hessian=hessian_np(bounce2_saddle),
        )

    err = abs(k_ml - lattice_ref) / abs(lattice_ref)
    finite_ess = all(np.isfinite(item["ESS"]) for item in trained.values())
    blocks_ok = all(
        item["block_scatter"] < max(5.0 * item["stderr"], 0.01 * abs(item["K"]))
        for item in trained.values()
    )
    passed = err < 0.05 and finite_ess and blocks_ok and not QUICK_TEST
    status = "PASS" if passed else ("QUICK_TEST_ONLY" if QUICK_TEST else "FAIL")

    mehler_ref = mehler_K00(T - 1j * ETA)
    print("\n" + "#" * 84, flush=True)
    print(f"RESULT  lambda={LAMBDA}, T={T}   using {n_saddles} saddles", flush=True)
    print("#" * 84, flush=True)
    print(f"K1 (zero saddle) = {k1:+.10f}", flush=True)
    if k3 is not None:
        print(f"K3               = {k3:+.10f}", flush=True)
    if k5 is not None:
        print(f"K5               = {k5:+.10f}", flush=True)
    print(f"K_ML             = {k_ml:+.10f}", flush=True)
    print(f"K_lattice        = {lattice_ref:+.10f}", flush=True)
    print(f"error vs lattice = {100 * err:.3f}%   status={status}", flush=True)
    if notes:
        print("note: " + " | ".join(notes), flush=True)

    row = {
        "lambda": LAMBDA,
        "T": T,
        "N": N,
        "eta": ETA,
        "run_1_bounce": RUN_1_BOUNCE,
        "run_3_bounces": RUN_3_BOUNCES,
        "run_5_bounces": RUN_5_BOUNCES,
        "n_saddles": n_saddles,
        "has_bounce": bounce_saddle is not None and "bounce" in trained,
        "has_bounce2": bounce2_saddle is not None and "bounce2" in trained,
        "err1": rel_err(k1, lattice_ref),
        "err3": rel_err(k3, lattice_ref),
        "err5": rel_err(k5, lattice_ref),
        "complex_rel_err_vs_lattice": err,
        "trivial_ESS_over_N": trained["trivial"]["ESS"],
        "bounce_ESS_over_N": trained["bounce"]["ESS"] if "bounce" in trained else _blank(),
        "bounce2_ESS_over_N": trained["bounce2"]["ESS"] if "bounce2" in trained else _blank(),
        "status": status,
        "note": " | ".join(notes),
        "run_directory": str(run_dir),
    }
    row.update(_z_fields("K1", k1))
    row.update(_z_fields("K3", k3))
    row.update(_z_fields("K5", k5))
    row.update(_z_fields("K_ML", k_ml))
    row.update(_z_fields("K_lattice", lattice_ref))
    row.update(_z_fields("K_mehler", mehler_ref))
    (run_dir / "results.json").write_text(json.dumps(row, indent=2, default=str), encoding="utf-8")
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return row


def read_done(path):
    rows = []
    done = set()
    if not path.exists():
        return rows, done
    with path.open("r", encoding="utf-8") as handle:
        for old in csv.DictReader(handle):
            rows.append(old)
            if old.get("status") in ("PASS", "FAIL", "QUICK_TEST_ONLY"):
                done.add((
                    round(float(old["lambda"]), 10),
                    round(float(old["T"]), 5),
                    old.get("run_1_bounce") == "True",
                    old.get("run_3_bounces") == "True",
                    old.get("run_5_bounces") == "True",
                ))
    return rows, done


def append_row(path, row):
    new_file = not path.exists() or path.stat().st_size == 0
    with path.open("a", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=ROW_FIELDS, extrasaction="ignore")
        if new_file:
            writer.writeheader()
        writer.writerow({key: row.get(key, "") for key in ROW_FIELDS})
        handle.flush()


lattice_table, _continuum = load_lattice()
comparison_csv = loop_dir / "multihimble_vs_lattice.csv"
all_rows, done = read_done(comparison_csv)
print(f"Already finished: {len(done)}  (lambda, T, switches) rows.", flush=True)
print(f"Table: {comparison_csv.resolve()}", flush=True)

jobs = JOBS if JOBS is not None else [(lam, t) for lam in LAMBDA_LIST for t in T_LIST]
for lam, t in jobs:
    LAMBDA = float(lam)
    t = float(t)
    print("\n" + "#" * 84, flush=True)
    print(f"LAMBDA = {LAMBDA}  T = {t}", flush=True)
    print("#" * 84, flush=True)
    key = job_key(LAMBDA, t)
    if key in done:
        print(f"skip lambda={LAMBDA} T={t} (already saved)", flush=True)
        continue
    print("\n" + "*" * 84, flush=True)
    print(f"T {t}   (lambda={LAMBDA})", flush=True)
    print("*" * 84, flush=True)
    lattice_key = (round(float(LAMBDA), 10), round(float(t), 5))
    try:
        row = run_one_time(t, lattice_table[lattice_key])
    except Exception:
        traceback.print_exc()
        print(f"FAILED lambda={LAMBDA} T={t}; continuing.", flush=True)
        continue
    all_rows.append(row)
    append_row(comparison_csv, row)
    done.add(key)
    print(f"Saved {len(done)} finished rows to {comparison_csv}", flush=True)

print(f"\nCompleted {len(done)} finished (lambda, T) rows.", flush=True)


Already finished: 6  (lambda, T, switches) rows.
Table: C:\xampp1\Quantum\runs\MAIN_Multihimble_N16\multihimble_vs_lattice.csv


NameError: name 'JOBS' is not defined

## 7. Table

One row per finished \((\lambda, T)\). `err1`, `err3`, and `err5` are the complex relative error of that bounce count against the lattice. A blank means that count was not trained.


In [ ]:
def cell_num(row, key):
    value = row.get(key, "")
    if value is None or value == "":
        return None
    return float(value)


def pct(value):
    if value is None:
        return f"{'':>8s}"
    return f"{100 * value:8.3f}%"


print(f"Saved comparison CSV: {comparison_csv.resolve()}")
print()
latest = {}
for row in all_rows:
    latest[(round(float(row["lambda"]), 10), round(float(row["T"]), 5))] = row

for lam in LAMBDA_LIST:
    print("=" * 118)
    print(f"lambda = {lam}")
    print("=" * 118)
    header = (
        f"{'T':>6s}  {'n':>2s}  {'err1':>9s}  {'err3':>9s}  {'err5':>9s}  "
        f"{'K_ML':>28s}  {'status':>8s}  note"
    )
    print(header)
    print("-" * len(header))
    for t in T_LIST:
        row = latest.get((round(float(lam), 10), round(float(t), 5)))
        if row is None:
            print(f"{t:6.2f}  not run")
            continue
        kml = complex(cell_num(row, "K_ML_re"), cell_num(row, "K_ML_im"))
        print(
            f"{float(row['T']):6.2f}  {int(float(row['n_saddles'])):2d}  "
            f"{pct(cell_num(row, 'err1'))}  {pct(cell_num(row, 'err3'))}  {pct(cell_num(row, 'err5'))}  "
            f"{kml:+.6f}  {row['status']:>8s}  {row.get('note', '')}"
        )
    print()


## 8. Plots

Navy is the lattice. The machine-learning lines are the bounce counts you turned on. A point with \(|K| > 8\) is left off the line so one divergence does not hide the rest.


In [ ]:
NAVY = "#1f4e79"
AMBER = "#d97706"
GREEN = "#15803d"
ORANGE = "#c2410c"


def series(rows, prefix):
    times, values = [], []
    for row in rows:
        re = cell_num(row, f"{prefix}_re")
        im = cell_num(row, f"{prefix}_im")
        if re is None or im is None:
            continue
        value = complex(re, im)
        times.append(float(row["T"]))
        values.append(value if abs(value) <= 8.0 else complex(np.nan, np.nan))
    return np.array(times), np.array(values)


nlam = len(LAMBDA_LIST)
fig, axes = plt.subplots(2, nlam, figsize=(5.2 * nlam, 7.2), squeeze=False, sharex=True)
for col, lam in enumerate(LAMBDA_LIST):
    rows = [
        latest[(round(float(lam), 10), round(float(t), 5))]
        for t in T_LIST
        if (round(float(lam), 10), round(float(t), 5)) in latest
    ]
    rows = sorted(rows, key=lambda row: float(row["T"]))
    ax_re, ax_im = axes[0, col], axes[1, col]
    if rows:
        t_lat, k_lat = series(rows, "K_lattice")
        ax_re.plot(t_lat, k_lat.real, color=NAVY, lw=1.8, label="lattice")
        ax_im.plot(t_lat, k_lat.imag, color=NAVY, lw=1.8, label="lattice")
        drawn = [("K1", RUN_1_BOUNCE, ORANGE, "1 bounce"),
                 ("K3", RUN_3_BOUNCES, AMBER, "3 bounces"),
                 ("K5", RUN_5_BOUNCES, GREEN, "5 bounces")]
        for prefix, enabled, color, label in drawn:
            if not enabled:
                continue
            times, values = series(rows, prefix)
            if len(times) == 0:
                continue
            ax_re.plot(times, values.real, color=color, marker="o", lw=1.4, ms=4, label=label)
            ax_im.plot(times, values.imag, color=color, marker="o", lw=1.4, ms=4, label=label)
    ax_re.axvline(np.pi, color="grey", ls=":", alpha=0.7)
    ax_im.axvline(np.pi, color="grey", ls=":", alpha=0.7)
    ax_re.set_title(rf"$\lambda={lam}$: Re $K$")
    ax_im.set_title(rf"$\lambda={lam}$: Im $K$")
    ax_re.set_ylabel("Re K")
    ax_im.set_ylabel("Im K")
    ax_im.set_xlabel("T")
    ax_re.grid(alpha=0.3)
    ax_im.grid(alpha=0.3)
    ax_re.legend(fontsize=8)

fig.suptitle(rf"1 / 3 / 5 bounces vs lattice   ($N={N}$, $\eta={ETA}$)", y=1.02)
fig.tight_layout()
summary_plot = loop_dir / "summary_1_3_5_vs_lattice.png"
fig.savefig(summary_plot, dpi=160, bbox_inches="tight")
plt.show()
print(f"Saved summary plot: {summary_plot.resolve()}")
print("Off the scale, not drawn: any point with |K| > 8.")
